In [1]:
#一、Installation（安装依赖）
#duckdb：数据库引擎本体（你要跑 SQL）
#duckdb-engine：把 DuckDB 接到 SQLAlchemy/Jupyter SQL 工具的桥梁
#jupysql：让你能用 %sql / %%sql 在 notebook 里直接写 SQL cell

%pip install duckdb duckdb-engine jupysql

Note: you may need to restart the kernel to use updated packages.


In [2]:
#二、导入
import duckdb
import pandas as pd

# Import jupysql Jupyter extension to create SQL cells
# 加载 JupySQL 的 Jupyter 扩展，让你可以用 %sql / %%sql 运行 SQL。
%load_ext sql

In [3]:
#三、配置 JupySQL 输出格式（让结果更好用）
#Set configurations on jupysql to directly output data to Pandas and to simplify the output that is printed to the notebook.
#在 jupysql 上设置配置，直接将数据输出到 Pandas，并简化打印到 notebook 的输出。

#查询结果自动变成 pandas DataFrame（更好看、更好用）。
%config SqlMagic.autopandas = True
#关闭一些“执行反馈/提示文本”，让输出更干净。
%config SqlMagic.feedback = False
#不在输出里显示连接信息（更简洁）
%config SqlMagic.displaycon = False

In [4]:
#四、Connecting to DuckDB
#Connect jupysql to DuckDB using a SQLAlchemy-style connection string. You may either connect to an in memory DuckDB, or a file backed db.
#使用 SQLAlchemy 风格的连接字符串将 JupySQL 连接到 DuckDB。

#用“连接字符串”连接到 内存数据库（数据存在内存，重启就没了）。
%sql duckdb:///:memory:

#如果你想把数据存成文件型数据库（持久化），就把路径换成你的 .db 文件路径。
# %sql duckdb:///path/to/file.db

#%sql：适合单行 SQL；%%sql：适合多行 SQL（一个 cell 写多行查询/建表语句）

In [5]:
#五、查看 DuckDB 扩展（extensions）列表
#查询 DuckDB 当前有哪些扩展、是否已安装/已加载。

In [6]:
%%sql
SELECT * FROM duckdb_extensions();

,extension_name,loaded,installed,install_path,description,aliases,extension_version,install_mode,installed_from
0,autocomplete,False,False,,Adds support for autocomplete in the shell,[],,NOT_INSTALLED,
1,aws,False,False,,Provides features that depend on the AWS SDK,[],,NOT_INSTALLED,
2,azure,False,False,,Adds a filesystem abstraction for Azure blob s...,[],,NOT_INSTALLED,
3,core_functions,True,True,(BUILT-IN),Core function library,[],v1.4.3,STATICALLY_LINKED,
4,delta,False,False,,Adds support for Delta Lake,[],,NOT_INSTALLED,
5,ducklake,False,False,,"Adds support for DuckLake, SQL as a Lakehouse ...",[],,NOT_INSTALLED,
6,encodings,False,False,,All unicode encodings to UTF-8,[],,NOT_INSTALLED,
7,excel,False,False,,Adds support for Excel-like format strings,[],,NOT_INSTALLED,
8,fts,False,False,,Adds support for Full-Text Search Indexes,[],,NOT_INSTALLED,
9,httpfs,False,True,/home/jovyan/.duckdb/extensions/v1.4.3/linux_a...,Adds support for reading and writing files ove...,"[http, https, s3]",9c7d349,REPOSITORY,core


In [7]:
#六、安装并加载 httpfs（关键：让你能直接读网页上的 CSV/Parquet）

#DuckDB's [httpfs extension](https://duckdb.org/docs/extensions/httpfs) allows parquet and csv files to be queried remotely over http. This is useful for querying large datasets without having to download them locally.
#DuckDB 的httpfs 扩展允许通过 HTTP 远程查询 parquet 和 csv 文件。这对于查询大型数据集非常有用，无需将其下载到本地。 

In [8]:
%%sql

INSTALL httpfs;
LOAD httpfs;

,Success


In [9]:
# 七、直接从网页读取 CSV（不落地建表）

In [10]:
%%sql
SELECT * FROM 'https://storage.googleapis.com/qm2/CASA0025/cities.csv';

,id,name,country,latitude,longitude,population
0,1,Bombo,UGA,0.58330,32.53330,75000
1,2,Fort Portal,UGA,0.67100,30.27500,42670
2,3,Potenza,ITA,40.64200,15.79900,69060
3,4,Campobasso,ITA,41.56300,14.65600,50762
4,5,Aosta,ITA,45.73700,7.31500,34062
...,...,...,...,...,...,...
1244,1245,Rio de Janeiro,BRA,-22.92502,-43.22502,11748000
1245,1246,Sao Paulo,BRA,-23.55868,-46.62502,18845000
1246,1247,Sydney,AUS,-33.92001,151.18518,4630000
1247,1248,Singapore,SGP,1.29303,103.85582,5183700


In [11]:
%%sql
SELECT * FROM 'https://storage.googleapis.com/qm2/CASA0025/countries.csv';

,id,Country,Alpha2_code,Alpha3_code,Numeric_code,Latitude,Longitude
0,1,Afghanistan,AF,AFG,4,33.0000,65.0
1,2,Albania,AL,ALB,8,41.0000,20.0
2,3,Algeria,DZ,DZA,12,28.0000,3.0
3,4,American Samoa,AS,ASM,16,-14.3333,-170.0
4,5,Andorra,AD,AND,20,42.5000,1.6
...,...,...,...,...,...,...,...
238,239,Wallis and Futuna,WF,WLF,876,-13.3000,-176.2
239,240,Western Sahara,EH,ESH,732,24.5000,-13.0
240,241,Yemen,YE,YEM,887,15.0000,48.0
241,242,Zambia,ZM,ZMB,894,-15.0000,30.0


In [12]:
#八、从网页 CSV 创建“真正的表”（后面 JOIN/聚合都用它）

In [13]:
%%sql 
CREATE TABLE cities AS SELECT * FROM 'https://storage.googleapis.com/qm2/CASA0025/cities.csv';

,Success


In [14]:
%%sql 
CREATE TABLE countries AS SELECT * FROM 'https://storage.googleapis.com/qm2/CASA0025/countries.csv';

,Success


In [17]:
%%sql
SELECT COUNT(*) AS n FROM cities;
SELECT COUNT(*) AS n FROM read_csv_auto('https://storage.googleapis.com/qm2/CASA0025/cities.csv');
SELECT * FROM read_csv_auto('https://storage.googleapis.com/qm2/CASA0025/cities.csv') LIMIT 5;

,id,name,country,latitude,longitude,population
0,1,Bombo,UGA,0.5833,32.5333,75000
1,2,Fort Portal,UGA,0.6710,30.2750,42670
2,3,Potenza,ITA,40.6420,15.7990,69060
3,4,Campobasso,ITA,41.5630,14.6560,50762
4,5,Aosta,ITA,45.7370,7.3150,34062


In [ ]:
%%sql 
FROM cities;
--Display the table content in the database.

In [ ]:
%%sql 
FROM countries;
--不是所有数据库都支持单独写 FROM table; 这种简写

In [ ]:
#九、The SQL SELECT statement

In [ ]:
%%sql 
SELECT * FROM cities;
--选并快速预览

In [ ]:
%%sql
SELECT * FROM cities LIMIT 10;
--前10行

In [ ]:
%%sql
SELECT name, country FROM cities LIMIT 10;
--部分+前10行

In [ ]:


#不同的值；去重
%%sql
SELECT DISTINCT country FROM cities LIMIT 10;

#count 
%%sql
SELECT COUNT(*) FROM cities;

#先去重再计数
%%sql
SELECT COUNT(DISTINCT country) FROM cities;

#最大值
%%sql
SELECT MAX(population) FROM cities;

#总值
%%sql
SELECT SUM(population) FROM cities;

#average
%%sql
SELECT AVG(population) FROM cities;

#order 字母排序
%%sql
SELECT * FROM cities ORDER BY country LIMIT 10;
 
#倒序
%%sql 
SELECT * FROM cities ORDER BY country ASC, population DESC LIMIT 10;

In [ ]:
#十、The WHERE Clause
#筛选
%%sql
SELECT * FROM cities WHERE country='USA'

%%sql
SELECT * FROM cities WHERE country='USA' OR country='CAN';

%%sql 
SELECT * FROM cities WHERE country='USA' AND population>1000000;

#not两种
SELECT * FROM cities
WHERE NOT country = 'USA';

SELECT * FROM cities
WHERE country <> 'USA';

SELECT * FROM cities
WHERE country IS NULL OR country NOT IN ('USA','CAN');

#like
#%=任意长度字符串
%%sql
SELECT * FROM cities WHERE country LIKE 'U%';

#以 A 结尾
%%sql
SELECT * FROM cities WHERE country LIKE '%A';

#包含
%%sql 
SELECT * FROM cities WHERE country LIKE '_S_';

#IN
%%sql
SELECT * FROM cities WHERE country IN ('USA', 'CAN');

#between
%%sql 
SELECT * FROM cities WHERE population BETWEEN 1000000 AND 10000000;

In [ ]:
#十一、SQL Joins

In [ ]:
%%sql 
SELECT COUNT(*) FROM cities;

In [ ]:
%%sql 
SELECT * FROM cities LIMIT 10;

In [ ]:
%%sql 
SELECT COUNT(*) FROM countries;

In [ ]:
%%sql 
SELECT * FROM countries LIMIT 10;

In [ ]:
%%sql
SELECT * FROM cities INNER JOIN countries ON cities.country = countries."Alpha3_code";
--11.1 NNER JOIN（只保留两边都匹配）
--加双引号表示“按原样精确匹配这个列名”。

In [ ]:
%%sql
SELECT name, cities."country", countries."Country" FROM cities INNER JOIN countries ON cities.country = countries."Alpha3_code";
--#只选需要的列（避免输出一堆重复列）

In [ ]:
%%sql
SELECT * FROM cities LEFT JOIN countries ON cities.country = countries."Alpha3_code";
--11.2 NNER JOIN（只保留两边都匹配）

In [ ]:
%%sql
SELECT * FROM cities LEFT JOIN countries ON cities.country = countries."Alpha3_code";
--11.3 SQL Right Join

In [ ]:
%%sql
SELECT * FROM cities FULL JOIN countries ON cities.country = countries."Alpha3_code";
--11.4 FULL JOIN（两边都保留）

In [ ]:
%%sql
SELECT country FROM cities
UNION 
SELECT "Alpha3_code" FROM countries;

--11.5 UNION（把两个查询结果“纵向合并”，并去重）

In [ ]:
#十二、Aggregation聚合

In [ ]:
%%sql

SELECT COUNT(name), country 
FROM cities 
GROUP BY country 
ORDER BY COUNT(name) DESC;
--12.1 order by

In [ ]:
%%sql

SELECT countries."Country", COUNT(name)
FROM cities
LEFT JOIN countries ON cities.country = countries."Alpha3_code"
GROUP BY countries."Country"
ORDER BY COUNT(name) DESC;
--代码（分组时把国家全名 join 进来；）

In [ ]:
%%sql 

SELECT COUNT(name), country
FROM cities
GROUP BY country
HAVING COUNT(name) > 40
ORDER BY COUNT(name) DESC;
--12.2 Having（对“分组后的结果”再筛选）
/* WHERE 是分组前筛行；
   HAVING 是分组后筛组（因为 WHERE 不能直接用聚合条件）*/